# SCU35 encrypted Ethernet payloads

Use the direct 100 Mb/s Ethernet link to send real host payloads through PMC AES-GCM and SHA hardware. The AES key is public test data; this is a functional demo, not a provisioned secure channel. No eFUSE or flash programming.

Build using `build.py`, then load the resulting `ethernet.pdi` into the exact SCU35 through Vivado Hardware Manager. Set the host interface to `192.168.10.1/24`. The board is `192.168.10.2`, UDP port 6000. `run_hardware.py` provides a separate automated test with UART capture and recovery. This notebook never programs hardware.

**Saved execution:** 2026-10-26T03:32:37.509160+00:00. Fresh offline verification of the recorded October 25 physical Ethernet capture. No new Ethernet hardware traffic or programming was performed by this execution. RUN_LIVE remains disabled.

In [1]:
from pathlib import Path
import sys, hashlib
import pandas as pd
for base in [Path.cwd(), *Path.cwd().parents]:
    candidate = base if (base / 'host.py').exists() else base / 'ethernet'
    if (candidate / 'host.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Open this notebook from the scu35 or ethernet directory')
from host import Client
RUN_LIVE = False  # Explicitly enable after loading the matching SCU35 PDI.

In [2]:
import json
recorded = candidate / 'evidence/run2/ethernet/results.json'
report = json.loads(recorded.read_text())
print('Recorded physical Ethernet run: October 25, 2026; not a new hardware run')
print(report['overall'], len(report['checks']), 'functional checks')
display(pd.DataFrame(report['benchmarks']))
from verify_capture import verify
verify(recorded)

Recorded physical Ethernet run: October 25, 2026; not a new hardware run
PASS 24 functional checks


,method,payload_bytes,packets,median_rtt_us,p95_rtt_us,sequential_payload_MBps
0,UDP plaintext echo,64,50,351.15,380.9,0.181530
1,UDP plaintext echo,256,50,466.80,526.1,0.543229
2,UDP plaintext echo,1024,50,986.55,1066.1,1.028711
3,PMC encrypt response,64,50,416.65,472.1,0.151189
4,PMC encrypt response,256,50,549.85,677.8,0.453854
5,PMC encrypt response,1024,50,1051.55,1120.4,0.965559
6,PMC decrypt + encrypt echo,64,50,474.15,542.5,0.132925
7,PMC decrypt + encrypt echo,256,50,607.40,665.5,0.416144
8,PMC decrypt + encrypt echo,1024,50,1124.00,1212.7,0.905112


OFFLINE_ETHERNET_CAPTURE_PASS: 20 exact payload/hash cases; bad-tag and replay response records; unique response nonces


{'valid_vectors': 20,
 'negative_vectors': 2,
 'scope': 'Cryptographic consistency of captured protocol bytes; capture hashes do not independently attest its physical origin.'}

## Send arbitrary data, verify encryption and decryption

Operation 1 sends a plaintext test request and returns an encrypted response. Operation 2 authenticates and decrypts an encrypted request, then sends an encrypted echo. Python verifies the GCM tag before using plaintext. The request/response round-trip clock excludes host request encryption and response decryption.

In [3]:
client = None
if RUN_LIVE:
    client = Client(state=Path('out/live/peer_state.json'))
    payload = bytes(range(256)) * 4
    plaintext, measurement, encrypted_packet = client.call(2, payload)
    assert plaintext == payload
    print('1024 host bytes authenticated/decrypted by SCU35 and returned in an encrypted response')
    display(pd.DataFrame([measurement]))
    print('Wire sample:', encrypted_packet[:80].hex())
else:
    print('Hardware cells disabled; set RUN_LIVE=True after loading the Ethernet PDI.')

Hardware cells disabled; set RUN_LIVE=True after loading the Ethernet PDI.


## Verify SHA3-256 and SHAKE-256 on host-supplied bytes

These requests use hardened SHA and return the 32-byte digest in an AES-GCM protected response.

In [4]:
if RUN_LIVE:
    for op, reference in [(5, hashlib.sha3_256(payload).digest()), (6, hashlib.shake_256(payload).digest(32))]:
        digest, measurement, _ = client.call(op, payload)
        assert digest == reference
        print('Exact digest match:', digest.hex())

## Reject modified tags and replayed requests

The board verifies a request tag before releasing plaintext. A successful authenticated peer session is pinned for this boot. Repeated requests with old sequence numbers are rejected. Reuse the same host state file throughout a board session; changing peer sessions requires restarting the board.

In [5]:
if RUN_LIVE:
    request = client.request(2, payload)
    client.exchange(request)
    _, replay, _ = client.exchange(request, expected_status=3)
    bad = bytearray(client.request(2, payload)); bad[-1] ^= 1
    _, invalid, _ = client.exchange(bytes(bad), expected_status=2)
    display(pd.DataFrame([replay, invalid]))

In [6]:
if client:
    client.close()
    client = None

For the complete multi-size benchmark, use `python host.py --out out/live --count 50` with the **same peer state file** (or restart the board first). Its report compares plaintext UDP echo, PMC encrypted responses, and PMC decrypt/encrypt echo. These are real network request/response measurements, distinct from the native resident-buffer processing benchmark.

When finished, restore the selected known-good PDI. The automated `run_hardware.py` does this in its recovery path. The notebook leaves the already loaded design running.